In [1]:
import pandas as pd
import numpy as np

#Load all tables

In [6]:
orders    = pd.read_csv('../data/raw/olist_orders_dataset.csv')
items     = pd.read_csv('../data/raw/olist_order_items_dataset.csv')
customers = pd.read_csv('../data/raw/olist_customers_dataset.csv')
products  = pd.read_csv('../data/raw/olist_products_dataset.csv')
sellers   = pd.read_csv('../data/raw/olist_sellers_dataset.csv')
payments  = pd.read_csv('../data/raw/olist_order_payments_dataset.csv')
reviews   = pd.read_csv('../data/raw/olist_order_reviews_dataset.csv')
geo       = pd.read_csv('../data/raw/olist_geolocation_dataset.csv')
cats      = pd.read_csv('../data/raw/product_category_name_translation.csv')

#Quick shape check

In [7]:
for name, df in [('orders',orders),('items',items),('customers',customers)]:
    print (f'{name}: {df.shape[0]:,} rows x {df.shape[1]} cols')

orders: 99,441 rows x 8 cols
items: 112,650 rows x 7 cols
customers: 99,441 rows x 5 cols


#Parse all timestamp columns at once

In [10]:
date_cols= ['order_purchase_timestamp', 'order_approved_at', 'order_delivered_carrier_date', 'order_delivered_customer_date', 'order_estimated_delivery_date']

for col in date_cols:
    orders[col] = pd.to_datetime(orders[col])

#Null audit 
print(orders.isnull().sum())
print(f'Null delivery dates: {orders.order_delivered_customer_date.isna().sum():,}')


#Keep only delivered orders
delivered=orders[orders['order_status']=='delivered'].copy()
print(f'Delivered orders: {len(delivered):,} out of {len(orders):,}')

order_id                            0
customer_id                         0
order_status                        0
order_purchase_timestamp            0
order_approved_at                 160
order_delivered_carrier_date     1783
order_delivered_customer_date    2965
order_estimated_delivery_date       0
dtype: int64
Null delivery dates: 2,965
Delivered orders: 96,478 out of 99,441


In [11]:
#Merge products with English category names
products=products.merge(cats, on='product_category_name', how='left')

#Build master flat table
master=(delivered.merge(items, on='order_id', how='left')
        .merge(customers, on='customer_id', how='left')
        .merge(sellers, on='seller_id', how='left')
        .merge(products, on='product_id', how='left')
        .merge(payments.groupby('order_id')['payment_value'].sum().reset_index(),on='order_id',how='left')
        )

#Add derived columns
master['order_month']=master['order_purchase_timestamp'].dt.to_period('M')
master['order_year']=master['order_purchase_timestamp'].dt.year
master['delivery_days']=(master['order_delivered_customer_date']-master['order_purchase_timestamp']).dt.days
master['revenue']=master['price']+master['freight_value']

print(f'Master table: {master.shape}')
master.to_csv('../data/processed/master.csv', index=False)

Master table: (110197, 35)
